# Capstone Project

Subtitle

Bob  
Jane

# Introduction

Something something, and then I state some citable facts \[@Zelner2022\].

# Background

## Pinot Noir is a key varietal for the Oregon wine industry

Among wine producers, Pinot Noir is known for being a very difficult grape due to its thin skin, tight grape clusters, early ripening, and sensitivity to heat. For many years, Pinot Noir production was limited to regions like Burgundy, France that had a narrow band of warm, sunny days during the growing season paired with a pronounced overnight cooldown—ideal conditions for the slow, gentle ripening required for great Pinot Noir.

In the 1960s, the winemaker David Lett set out to find a region outside of Burgundy where he could successfully cultivate Pinot Noir vines. He settled in Oregon’s Willamette Valley, which sits on the same latitude as Burgundy (close to the 45th parallel) and has been assessed as sharing many of the same climate characteristics relevant to Pinot Noir viticultural site selection \[@jonesHellman2003\], including the day-night temperature swing that is vital to Pinot Noir’s unique character. In 1979, Lett entered his Willamette Valley Pinot Noir in the Gault-Millau French Wine Olympiad, where he placed among the top Burgundies. This was the first real signal that Oregon could compete at the same level as the region that had produced the best Pinot Noirs for hundreds of years \[@oregonwine2025willamette\]. The Willamette Valley became an official American Viticultural Area (AVA) in 1983 and has since become one of the most recognized Pinot Noir regions in the world, a status widely attributed to its warm-day, cool-night climate pattern \[@oregonwine2025willamette; @sommselect2026\].

# The diurnal range effect

This unique climate pattern is referred to as *diurnal temperature range*. Warm days drive photosynthesis and sugar accumulation in the grapes, while cool nights slow the grape’s respiration rate, giving time for the acids, color compounds, and aromatic molecules to further develop rather than burning off overnight. Without this nightly cooldown, grapes ripen too fast and produce wines that taste “flat”—high in sugar without the balance of acidity and aromatic complexity that define great Pinot Noir \[@kliewer1972; @ajev2012dtr\].

An ideal diurnal range for Pinot Noir ripening is generally considered to be in the neighborhood of 15–20°C (~60-70°F). Controlled field and growth-chamber experiments on Pinot Noir have shown that raising the diurnal temperature range around the onset of ripening (*véraison*) alters organic acid metabolism and can inhibit color and phenolic accumulation in the grape \[@frontiers2025bunchheating; @cohen2012jexpbot\]. While compression of the diurnal range has been shown to hasten ripening and shift how flavors develop in wine grapes broadly \[@ajev2012dtr\], targeted studies on Malbec, Merlot, and Pinot Noir have found that diurnal range sensitivity is not universal across varieties and is particularly pronounced in Pinot Noir \[@pmc2022highTemp\].

When we consider the impact of climate on wine production, particularly in Oregon where Pinot Noir is the dominant variety, it is not just a question of whether summers are getting hotter *overall*, but rather what happens when *nights warm faster than days*. This is a documented feature of twentieth- and twenty-first-century climate shifts globally, generally attributed to changes in cloud cover trapping heat overnight \[@ucs2022nights; @liu2024dtr\]. This is exactly the pattern we observe in Oregon’s climate record: a real warming signal in growing degree days and heat stress, alongside shrinking day-night temperature swings during ripening \[@ucs2022nights; @liu2024dtr\].

## Prior research on Oregon and Pinot Noir climate sensitivity

Climate research specific to Oregon wine has, to date, focused primarily on phenology and single-variety climate suitability rather than on production outcomes directly. Multi-decade climate analyses have documented measurable statewide warming, pushing the Willamette Valley from sitting at the cool edge of Pinot Noir suitability in the 1960-70s to sitting centered within that suitability range today \[@jones2005\]. Work presented at the Oregon Wine Symposium has projected temperature-driven shifts in Oregon varietal suitability and harvest timing through 2100, pointing toward a potential reshuffling of the state’s variety composition as the climate continues to warm \[@skahill2025\]. Separately, phenological modeling published in the *American Journal of Enology and Viticulture* has modeled shifts in the timing of the Willamette Valley’s Pinot Noir growing season, reinforcing the variety’s particular sensitivity to the region’s climate \[@delelee2025\].

We draw our general climate-threshold framework (frost days, growing degree days, heat stress days) from a global synthesis of viticulture climate impacts \[@vanleeuwen2024\]. This project differs from that body of work in that, rather than modeling suitability or phenology on their own, we integrate 38 years of statewide production records directly with climate data to ask whether the warming trend has affected *production outcomes*, not just suitability.

## Filling an availability gap for Oregon wine data

Our motivation for this project is also driven by a genuine gap in publicly available data for Oregon wines. Due to the fact that 90% of U.S. wine grape acreage sits in California and Washington, USDA’s National Agricultural Statistics Service (NASS) discontinued collecting Oregon-specific grape data beginning in 2011 \[@nass2011gap\]. While similar data continued to be collected by researchers in Oregon, this information existed largely as scanned PDF reports rather than structured data. This project’s data engineering work is built in part to close that gap by producing a structured, climate-linked Oregon wine dataset that does not currently exist from any single public source.

# Data

## Data sources

**Oregon wine production and pricing** data comes from the Oregon Vineyard and Winery Annual Reports, conducted since 1981 by various research groups and currently maintained by the University of Oregon \[@oregonwinecensus\]. Reports prior to 2017 exist only as scanned & digital PDF documents and were ingested using AWS Textract; later years were ingested directly from structured spreadsheets.

**Climate data** was retrieved from the PRISM Climate Group, maintained by Oregon State University. PRISM provides daily gridded temperature and precipitation observations across the continental United States \[@prism2026\]. We retrieved daily observations from 1981-2024 across all 23 federally-recognized AVAs in Oregon. Zonal statistics were produced for each AVA by averaging the raster grid within each AVA boundary using the Python rasterstats library. AVA boundaries were defined by shapefiles obtained from the Alcohol and Tobacco Tax and Trade Bureau (TTB), which were converted to GeoJSON prior to their use in the ingestion script.

From the daily climate data, we derived a set of features chosen specifically for their relevance to viticulture, rather than working with raw daily temperature and precipitation directly.

| Feature | Definition |
|------------------------------------|------------------------------------|
| Frost risk days | Count of days below freezing, annually and during the April–May bud-break window |
| Coldest spring night | Minimum recorded temperature during spring bud-break |
| Growing degree days (GDD) | Accumulated heat above a 10°C (50°F) base, April–September (Winkler index), plus a April–October variant and a July 15–October 15 “véraison-to-harvest” window |
| Heat stress days | Count of days with a maximum temperature above 35°C (95°F) during the growing season |
| Vapor pressure deficit (VPD) | A measure of atmospheric dryness / evaporative demand, computed for summer (July–August) and ripening (August–September) windows |
| Diurnal temperature range (ripening) | Average day-night temperature swing during August–September, the period when grapes develop their sugar, color, and acidity |
| Harvest precipitation | Total precipitation in October, and in the September–October window |

## Data engineering pipeline

Climate and production data were ingested on a recurring Amazon Web Services (AWS) batch pipeline (one task per AVA), computed into region- and statewide-level summaries, and stored in a PostgreSQL database hosted on Railway. AWS was chosen specifically for the production-data side of the pipeline, as reconstructing tables from PDF reports required routing each report era through separate Textract-based parsers due to varied report formats, column layouts, and label conventions. For the climate data, daily PRISM rasters were cached prior to computing zonal statistics, which were then written to the database via an idempotent upsert. Running this pipeline on AWS Batch / Fargate Spot allowed each AVA to be processed as a parallel, independently retriable job, though Spot interruptions during the original run caused some days to be missed; these were identified and backfilled in a final pass before the analysis tables were built.

Two analysis-ready tables were constructed on top of the raw data:

| Panel | Coverage | Grain | Purpose |
|------------------|------------------|------------------|------------------|
| Regional panel | 1987-2024 | One row per region x variety x year | Yield modeling and region-level comparisons |
| Statewide panel | 1987-2024 | One row per variety x year | Long-run varietal-mix analysis where regional detail is unnecessary |

Both panels begin in 1987, rather than 1981, because of limitations in the production data. While the full PRISM climate dataset was available from 1981 for all AVAs, the underlying wine census reports for 1981-1986 do not contain the area-by-variety breakdown tables needed to reconstruct regional variety data. These reports *do* contain bearing and non-bearing acreage by county, which could in principle be aggregated to the region level for an acreage-only (not variety-level) time series; we did not pursue this, and instead clipped both panels to 1987 onward to keep the regional and statewide tables consistent.

Region-variety combinations were filtered to those with at least 20 years of historical data and zero-production / zero-harvest rows were dropped. This reduced the original ~1,300 region-variety-year observations to ~990 in the regional panel.

The analysis tables were exported to Parquet files, which are used as the final data references for this report to preserve reproducibility. Our long-term goal is to make the underlying database publicly available, but this is a continued work in progress.

## Ethical considerations

A few ethical considerations shaped how we approached this dataset:

1.  **Regional underrepresentation:** Underlying wine census data is reported by region, not by individual vineyard or winery, and is a voluntary industry survey rather than a full census. We recognize that this led to underrepresentation for some counties in some years.
2.  **Diversity and inclusion gaps among Oregon wine producers:** Because there are very few minority-owned wineries in Oregon—our research turned up only two Native-owned wineries statewide—there is a real possibility that patterns related to these ownership demographics are invisible in this dataset. The effect of climate may have a disproportionate impact on some groups over others, but data was never collected that could surface these patterns.
3.  **Impact of immigration on Oregon wine production:** Recent immigration policy has affected farmworkers and their families in ways that are impossible to quantify from production statistics alone. The truth is that Oregon wine production relies heavily on seasonal immigrant labor, and the raw production numbers do not tell the human story of these populations who are increasingly at risk in American society.

We treat this dataset as a record of production and climate outcomes only, not as a complete picture of the people who participate in the industry that these outcomes describe.

## Data quality caveats

We would like to provide transparency with respect to some key tradeoffs in our data collection:

- **Pre-2002 regional variety data is reconstructed, not directly reported.** From 1987–2001, the industry reports at the statewide level did not break variety out by region; those years were reconstructed by aggregating county-level figures up to reporting regions. North Willamette Valley acreage, for example, is undercounted by roughly 15–20% for years before 1999 because the source area tables for that era only list major counties. This undercount only affects the variety-level breakdown for that region; the “all varieties” total is accurate. We treat pre-2002 regional findings with corresponding caution throughout this report and do not present county-aggregated reconstructions as precise counts.
- **Regional and statewide panels begin in 1987, not 1981, despite full climate coverage back to 1981**: As described above, this is driven entirely by gaps in the wine production data source tables, not by any limitation in the climate data.
- **Price per ton (`PRI_01`) is only available at the region level from 2010 onward.** This is a genuine limitation of the source data and constrains how far back any price-based analysis can go. Additionally, because price is affected by many factors, of which climate is only one, our analysis centers mainly around production data rather than price data.
- **The wine census is self-reported and voluntary, not a full census.** We treat year-to-year volatility in the raw production series with more caution than the smoothed, multi-year trends.

None of these caveats change our core findings, which are drawn primarily from long-term trends and the more reliable 2002–2024 window and from the climate data which has no comparable gaps. PRISM provides complete daily coverage for all 23 AVAs across the full 1981–2024 period.

# Analysis

In [ ]:
#| include: false
library(arrow)


Attaching package: 'arrow'

The following object is masked from 'package:utils':

    timestamp

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     

── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ lubridate::duration() masks arrow::duration()
✖ dplyr::filter()       masks stats::filter()
✖ dplyr::lag()          masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Attaching package: 'scales'

The following object is masked from 'package:purrr':

    discard

The following object is masked from 'package:readr':

    col_factor

Loading required package: zoo


Attaching package: 'zoo'

The following objects are masked from 'package:base':

    as.Date, as.Date.numeric

Loading required package: sandwich


Attaching package: 'strucchange'

The following object is masked from 'package:stringr':

    boundary

Loading required package: carData


Attaching package: 'car'

The following object is masked from 'package:dplyr':

    recode

The following object is masked from 'package:purrr':

    some

## Oregon’s growing reliance on Pinot Noir

In [ ]:
#| echo: false
#| warning: false
#| message: false
#| fig-cap: "Figure 1: Statewide grape tonnage by variety, 1987–2024"
variety_share <- statewide %>%
  filter(!is.na(production_share_pct)
    , !is_zero_production
    , variety_label != "All"
    , yr >= 1987) %>%
  mutate(variety_group = if_else(variety_label %in% key_varieties, variety_label, "Other")) %>%
  group_by(yr, variety_group) %>%
  summarise(production_tons = sum(production_tons, na.rm = TRUE), .groups = "drop") %>%
  group_by(yr) %>%
  mutate(production_share_pct = production_tons / sum(production_tons, na.rm = TRUE) * 100) %>%
  ungroup()

labels_df <- variety_share %>%
  group_by(variety_group) %>%
  filter(yr == max(yr)) %>%
  ungroup()

variety_colors <- c(
  "Pinot Noir" = "#b2182b"
  , "Pinot Gris" = "#4393c3"
  , "Chardonnay" = "#d6604d"
  , "Riesling/White Riesling" = "#d4a017"
  , "Other" = "#999999"
)

variety_share %>%
  mutate(is_pinot = variety_group == "Pinot Noir") %>%
  ggplot(aes(yr, production_share_pct, color = variety_group, alpha = is_pinot, linewidth = is_pinot)) +
  geom_line(alpha = 0.2, linewidth = 0.4) +
  geom_smooth(method = "loess", se = FALSE, span = 0.4, aes(linewidth = is_pinot)) +
  geom_text_repel(
      data = labels_df, aes(x = yr, y = production_share_pct, label = variety_group, color = variety_group)
      , nudge_x = 1
      , hjust = 0
      , size = 3.2
      , segment.color = "grey70"
      , segment.size = 0.3
      , direction = "y"
      , show.legend = FALSE
      , inherit.aes = FALSE
    ) +
  scale_color_manual(values = variety_colors) +
  scale_linewidth_manual(values = c(`TRUE` = 1.4, `FALSE` = 0.8)) +
  scale_alpha_manual(values = c(`TRUE` = 1, `FALSE` = 0.7)) +
  scale_x_continuous(limits = c(1987, 2028), breaks = seq(1987, 2024, 5)) +
  labs(title = "Pinot Noir has come to define Oregon wine"
       , subtitle = "Share of statewide tonnage by variety, 1987–2024"
       , x = NULL, y = "Production share (%)") +
  theme_minimal() +
  theme(legend.position = "none"
        , plot.title = element_text(size = 16, face = "bold")
        , plot.subtitle = element_text(size = 10, color = "grey40"))

`geom_smooth()` using formula = 'y ~ x'

Figure 1 demonstrates Pinot Noir’s growing share of statewide tonnage, from roughly 15% in 1990 to around 30% by 2020. This trend has continued to build, while other anchor varieties like Chardonnay and Riesling have declined sharply over the same period. Pinot Gris is one variety that counteracts this trend, growing from a negligible share in 1990 to a stable ~8% by 2010.

However, the dramatic shift toward Pinot Noir has remained a dominant trend, suggesting that Oregon’s wine industry has increasingly concentrated on climate-sensitive Pinot varieties over exactly the same decades that have coincided with a pronounced shift in the region’s key climate indicators.

## Evidence of a warming climate

In [ ]:
#| echo: false
#| warning: false
#| message: false
#| fig-cap: "Figure 2: Growing degree day (GDD) anomaly relative to the 38-year average, overlaid with a 10-year rolling average"
sw_climate <- statewide %>%
  distinct(
    yr
    , gdd_apr_sep
    , gdd_apr_oct
    , frost_days_annual
    , frost_days_spring
    , heat_stress_days
    , summer_tmax_mean
    , oct_ppt_mm
    , annual_ppt_mm
    , vpd_max_summer
    , diurnal_range_ripening
    ) %>%
  arrange(yr) %>%
  mutate(
    gdd_rolling10 = zoo::rollmean(gdd_apr_sep, 10, fill = NA, align = "right")
    , gdd_anom = gdd_apr_sep - mean(gdd_apr_sep, na.rm = TRUE)
    , gdd_rolling10_anom = gdd_rolling10 - mean(gdd_apr_sep, na.rm = TRUE)
    )

# Structural-break test on the raw GDD series
gdd_ts <- ts(sw_climate$gdd_apr_sep, start = min(sw_climate$yr))
bp_fit <- breakpoints(gdd_ts ~ 1)
break_idx  <- bp_fit$breakpoints[1]
break_year <- sw_climate$yr[break_idx]

ggplot(sw_climate, aes(yr, gdd_anom)) +
  geom_col(aes(fill = gdd_anom > 0), width = 0.7) +
  geom_line(aes(y = gdd_rolling10_anom), color = "black", linewidth = 1) +
  geom_hline(yintercept = 0, color = "grey30", linewidth = 0.4) +
  scale_fill_manual(values = c(`TRUE` = "#b2182b", `FALSE` = "#2166ac"), guide = "none") +
  geom_vline(xintercept = break_year, linetype = "dashed", color = "grey40", linewidth = 0.6) +
  labs(title = "Oregon's growing season is getting warmer"
       , subtitle = "Growing Degree Days (GDD) anomaly from the 1987–2024 mean"
       , x = NULL, y = "GDD (Apr–Sep) anomaly") +
  theme_minimal() +
  theme(legend.position = "none"
      , plot.title = element_text(size = 16, face = "bold")
      , plot.subtitle = element_text(size = 10, color = "grey40"))

(`geom_line()`).

Growing degree days (GDDs) show a clear and sustained upward trend, with a level shift identified by a structural-break test around 2012.

In [ ]:
#| echo: false
#| warning: false
#| message: false
#| fig-cap: "Figure 3: Six viticulture-relevant climate indicators, statewide average, 1987–2024, each with a linear trend and 95% confidence interval"
sw_climate %>%
  select(yr
         , `Frost risk days` = frost_days_spring
         , `GDD (Apr-Sep)` = gdd_apr_sep
         , `Peak summer temperature (°C)` = summer_tmax_mean
         , `Heat stress days` = heat_stress_days
         , `Diurnal ripening temperature range (°C)` = diurnal_range_ripening
         , `October precipitation (mm)` = oct_ppt_mm
         ) %>%
  pivot_longer(-yr, names_to = "indicator", values_to = "value") %>%
  mutate(indicator = factor(indicator, levels = c(
    "Frost risk days"
    , "GDD (Apr-Sep)"
    , "Peak summer temperature (°C)"
    , "Heat stress days"
    , "Diurnal ripening temperature range (°C)"
    , "October precipitation (mm)"
  ))) %>%
  ggplot(aes(yr, value)) +
  geom_line(color = "grey60", alpha = 0.7) +
  geom_smooth(method = "lm", se = TRUE, color = "#b2182b", fill = "#b2182b", alpha = 0.15) +
  facet_wrap(~indicator, scales = "free_y", ncol = 2) +
  labs(title = str_wrap("Diurnal ripening range is a notable exception among Oregon's viticultural indicators", width = 60)
       , subtitle = "Oregon viticultrual indicators by season; linear trend lines with 95% confidence intervals, 1987-2024"
       , x = NULL, y = NULL) +
  theme_minimal() +
  theme(legend.position = "none"
      , plot.title = element_text(size = 16, face = "bold")
      , plot.subtitle = element_text(size = 10, color = "grey40"))

`geom_smooth()` using formula = 'y ~ x'

Figure 3 summarizes six key viticultural climate indicators. Growing degree days, peak summer temperature, and heat stress days are all trending up with tight confidence intervals, demonstrating a warming signal is both real and steep. Spring frost days are essentially flat, a meaningful non-finding since it means the warming trend has not reduced the risk of a damaging frost event during spring bud break, even as summer heat accumulation climbs. October precipitation trends slightly upward but noisily, with no strong signal either way.

The sixth indicator—ripening-season diurnal temperature range—is the one metric that moves in the opposite direction from the rest, and it does so specifically because it isn’t a simple byproduct of an overall upward heat trend. This finding is most directly relevant to Oregon’s growing dependence on Pinot Noir, since Pinot Noir relies on a wide day-night temperature swing during ripening to retain acidity and develop aromatic complexity—a mechanism we discussed in [Background](#background).

## How the climate features relate

Several of these climate measures are closely related to aspects of the same underlying warming trend.

In [ ]:
#| echo: false
#| warning: false
#| message: false
#| fig-cap: "Figure 4: Correlation matrix of annual climate features, statewide, 1987–2024"
# Aggregate to one row per year (mean across regions) before computing correlations
ry <- regional %>%
  distinct(
    region_label
    , yr
    , gdd_apr_sep
    , gdd_veraison_harvest
    , frost_days_spring
    , heat_stress_days
    , coldest_spring_night_c
    , oct_ppt_mm
    , sep_oct_ppt_mm
    , gs_ppt_mm
    , vpd_max_summer
    , diurnal_range_ripening
    )

climate_yearly <- ry %>%
  group_by(yr) %>%
  summarise(across(
    c(gdd_apr_sep, gdd_veraison_harvest, frost_days_spring, heat_stress_days
      , coldest_spring_night_c, oct_ppt_mm, sep_oct_ppt_mm, gs_ppt_mm
      , vpd_max_summer, diurnal_range_ripening)
    , mean, na.rm = TRUE
  ), .groups = "drop")

ℹ In argument: `across(...)`.
ℹ In group 1: `yr = 1987`.
Caused by warning:
! The `...` argument of `across()` is deprecated as of dplyr 1.1.0.
Supply arguments directly to `.fns` through an anonymous function instead.

  # Previously
  across(a:b, mean, na.rm = TRUE)

  # Now
  across(a:b, \(x) mean(x, na.rm = TRUE))

Figure 4 identifies three clusters of variables that appear to measure largely the same underlying phenomenon from different angles, rather than independent drivers:

1.  **Spring frost risk:** Frost risk days and coldest spring night are correlated at -0.91, which makes sense given one is a temperature reading and the other is a threshold-count derived from it.
2.  **Summer heat accumulation:** Heat stress days and summer VPD are both correlated above 0.6 with all the GDD variables.
3.  **Fall precipitation:** Precipitation measures move together above 0.4.

We also note that diurnal ripening temperature range is negatively correlated with nearly every heat-related feature, consistent with the pattern described in [Background](#background), where nights warm faster than days as the season heats up overall, compressing the day-night swing. Additionally, growing-season precipitation is negatively correlated with GDD Apr-Sep (-0.11), indicating that hotter years also tend to be drier years overall.

To resolve which variables to retain for modeling, we turn to variance inflation factors (VIF) to capture multi-variable overlap.

In [ ]:
#| echo: false
#| warning: false
#| message: false
# Setup
set.seed(42)
plot_vif <- function(vif_vals) {
  data.frame(Variable = names(vif_vals), VIF = round(unname(vif_vals), 2)) %>%
    mutate(Variable = dplyr::recode(Variable
      , "frost_days_spring" = "Frost risk days"
      , "coldest_spring_night_c" = "Coldest spring night"
      , "gdd_veraison_harvest" = "GDD Veraison-Harvest"
      , "gdd_apr_sep" = "GDD Apr-Sep"
      , "heat_stress_days" = "Heat stress days"
      , "vpd_max_summer" = "Summer VPD"
      , "diurnal_range_ripening" = "Diurnal temp range"
      , "gs_ppt_mm" = "Growing season precip"
      , "sep_oct_ppt_mm" = "Sep-Oct precip"
      , "oct_ppt_mm" = "Oct precip"
    )) %>%
    arrange(desc(VIF)) %>%
    knitr::kable(caption = "Variance Inflation Factors", row.names = FALSE)
}

ry_clean <- ry %>%
  filter(!region_label %in% c("Eastern Oregon", "Other Region", "Other Oregon"))

vif_df <- ry_clean %>%
  mutate(dummy_outcome = rnorm(n()))

# VIF - all variables
vif_model <- lm(dummy_outcome ~ gdd_apr_sep
  + frost_days_spring
  + coldest_spring_night_c
  + gdd_veraison_harvest
  + heat_stress_days
  + vpd_max_summer
  + diurnal_range_ripening
  + sep_oct_ppt_mm
  + oct_ppt_mm
  , data = vif_df
  )
vif_vals <- car::vif(vif_model)
plot_vif(vif_vals)

  Variable                  VIF
  ---------------------- ------
  Sep-Oct precip           8.11
  Oct precip               6.75
  Summer VPD               6.43
  Coldest spring night     4.38
  GDD Apr-Sep              4.13
  Frost risk days          3.82
  Heat stress days         2.90
  GDD Veraison-Harvest     2.34
  Diurnal temp range       2.27

  : Variance Inflation Factors


Sep-Oct precipitation and Oct precipitation come out highest, confirming we should not include both in the model. We elected to drop Sep-Oct precipitation, as we recognize Oct precipiatation specifically as an important botrytis / harvest-timing risk.

Surprisingly, Summer VPD is *higher* than GDD Apr-Sep (6.43 vs. 4.13), even though the correlation matrix showed VPD’s strongest pairwise correlation was with GDD. This tells us Summer VPD is well-explained by some *combination* of GDD Apr-Sep, heat stress days, and ripening temp range together, rather than by a strong individual pairwise correlation with any one of them. We elect to drop GDD Apr-Sep and keep Summer VPD, as it is the more direct measure of vine water stress.

In [ ]:
#| echo: false
#| warning: false
#| message: false
# VIF - final features
vif_model_2 <- lm(dummy_outcome ~ gdd_veraison_harvest 
  + frost_days_spring 
  + coldest_spring_night_c
  + heat_stress_days 
  + vpd_max_summer 
  + diurnal_range_ripening
  + oct_ppt_mm 
  , data = vif_df
  )
vif_vals <- car::vif(vif_model_2)
plot_vif(vif_vals)

  Variable                  VIF
  ---------------------- ------
  Coldest spring night     4.18
  Frost risk days          3.81
  Summer VPD               3.71
  Heat stress days         2.66
  GDD Veraison-Harvest     2.12
  Diurnal temp range       1.31
  Oct precip               1.15

  : Variance Inflation Factors


All remaining VIFs fall under a conservative threshold of VIF\<5. Our retained predictor set going into the modeling stage is:

| Feature | Definition |
|------------------------------------|------------------------------------|
| Frost risk days | Count of days below freezing during the April–May bud-break window |
| Coldest spring night | Minimum recorded temperature during spring bud-break |
| GDD (Véraison–Harvest) | Accumulated heat above a 10°C (50°F) base during July 15–October 15 “véraison-to-harvest” window |
| Heat stress days | Count of days with a maximum temperature above 35°C (95°F) during the growing season |
| Summer VPD | A measure of atmospheric dryness / evaporative demand during July–August |
| Diurnal temperature range (ripening) | Average day-night temperature swing during August–September, the period when grapes develop their sugar, color, and acidity |
| Harvest precipitation (October) | Total precipitation in October |

Note that GDD Apr-Sep itself remains central to the *descriptive* climate trends in Figures 2–3 above; it is excluded only from the predictive feature set, where its information is substantially redundant with the other retained heat measures.

# Results

In [ ]:
#| include: false
library(arrow)
library(tidyverse)
library(ggplot2)
library(scales)

regional  <- read_parquet("data/analysis_panel_regional.parquet")
statewide <- read_parquet("data/analysis_panel_statewide.parquet")

regional <- regional %>%
  mutate(yield_derived = production_tons / acres_harvested,
         variety_label = str_trim(variety_label))
statewide <- statewide %>%
  mutate(yield_derived = production_tons / acres_harvested,
         variety_label = str_trim(variety_label))

key_varieties <- c("Pinot Noir", "Pinot Gris", "Chardonnay", "Riesling/White Riesling")

## Production and yield have, so far, kept climbing

In [ ]:
#| fig-cap: "Figure 4: Total Oregon wine grape production, 1987–2024, with separate smoothed trends fit around 2002 reporting methodology shift"
statewide_total <- statewide %>%
  filter(variety_label == "All") %>%
  select(yr, production_tons) %>%
  arrange(yr)

ggplot(statewide_total, aes(x = yr, y = production_tons)) +
  geom_line(color = "gray70", linewidth = 0.5, alpha = 0.6) +
  geom_smooth(data = statewide_total %>% filter(yr <= 2001), method = "loess", se = FALSE, color = "#8B1E3F", linewidth = 1.3) +
  geom_smooth(data = statewide_total %>% filter(yr >= 2002), method = "loess", se = FALSE, color = "#8B1E3F", linewidth = 1.3) +
  annotate("segment", x = 2020, xend = 2020, y = 0, yend = max(statewide_total$production_tons, na.rm = TRUE) * 0.72
           , linetype = "dashed", color = "gray40", linewidth = 0.5) +
  annotate("text", x = 2020, y = max(statewide_total$production_tons, na.rm = TRUE) * 0.68
           , label = str_wrap("2020: wildfire smoke taint", width = 15), hjust = 0, size = 3.2, color = "gray30", lineheight = 0.9) +
  scale_y_continuous(labels = comma) +
  labs(title = "Total Oregon wine grape production, 1987–2024",
       subtitle = "Smoothed trend, fit separately before/after 2002 methodology shift"
       , x = NULL, y = "Production (tons)") +
  theme_minimal()

`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'

(`stat_smooth()`).

Figure 4 demonstrates roughly tenfold growth in statewide wine production since the late 1980s, tracking the industry’s broader economic growth. We note a clear interruption in 2020, corresponding to a season with severe wildfire smoke taint that resulted in a substantial share of unusable fruit among that year’s harvest. This represents a discrete event rather than a gradual climate-trend effect.

In [ ]:
#| fig-cap: "Figure 5: Production-weighted yield by **variety**, statewide, 1987–2024"
weighted_yield_variety <- regional %>%
  filter(!is.na(production_tons), !is.na(acres_harvested), acres_harvested > 0
         , production_tons > 0, variety_label != "All") %>%
  mutate(variety_group = if_else(variety_label %in% key_varieties, variety_label, "Other")) %>%
  group_by(yr, variety_group) %>%
  summarise(total_prod = sum(production_tons, na.rm = TRUE)
            , total_acres = sum(acres_harvested, na.rm = TRUE)
            , weighted_yield = total_prod / total_acres
            , .groups = "drop")

ggplot(weighted_yield_variety, aes(yr, weighted_yield, color = variety_group)) +
  geom_line() +
  geom_vline(xintercept = 2020, linetype = "dashed", color = "grey40") +
  annotate("text", x = 2020.3, y = Inf, label = "2020 fires", hjust = 0, vjust = 2, color = "grey40", size = 3.5) +
  scale_color_manual(values = c("Pinot Noir" = "#b2182b", "Pinot Gris" = "#4393c3"
                                 , "Chardonnay" = "#d6604d", "Riesling/White Riesling" = "#d4a017"
                                 , "Other" = "#999999")) +
  labs(title = "Yield by variety, statewide",
       subtitle = "Production-weighted yield (total tons ÷ total harvested acres) — normalizes for acreage changes"
       , x = NULL, y = "Yield (tons per acre)", color = "Variety") +
  theme_minimal()

In [ ]:
#| fig-cap: "Figure 6: Production-weighted yield by **region**, 2002–2024"
weighted_yield_region <- regional %>%
  filter(!is.na(production_tons), !is.na(acres_harvested), acres_harvested > 0
         , production_tons > 0, yr >= 2002
         , !region_label %in% c("Eastern Oregon", "Other Region", "Other Oregon")) %>%
  group_by(yr, region_label) %>%
  summarise(total_prod = sum(production_tons, na.rm = TRUE)
            , total_acres = sum(acres_harvested, na.rm = TRUE)
            , weighted_yield = total_prod / total_acres
            , .groups = "drop")

ggplot(weighted_yield_region, aes(yr, weighted_yield, color = region_label)) +
  geom_line(linewidth = 0.8) +
  geom_vline(xintercept = 2020, linetype = "dashed", color = "grey40") +
  scale_color_brewer(palette = "Set1") +
  labs(title = "Yield by region, 2002–2024"
       , subtitle = "Production-weighted yield (total tons ÷ total harvested acres)"
       , x = NULL, y = "Yield (tons per acre)", color = "Region") +
  theme_minimal()

Figure 5 shows a dramatic dip in Pinot Noir yield in 2020—from roughly 62,000 to 40,000 tons statewide—while Pinot Gris and Chardonnay barely register the disruption. This selective impact is consistent with wildfire smoke taint, which binds more readily to red wine grapes because of their extended skin-contact fermentation process than to white wine production; anecdotally, 2020 also drove some producers toward “white Pinot” as a workaround, itself a sign of the industry’s adaptive capacity.

All varieties rebound to new all-time highs within three years. Indeed, outside of that one disrupted year, yield for Pinot Noir and the other key varieties has held steady or trended upward across regions, and the same holds when the data is cut by region instead of variety in Figure 6. The historical pattern demonstrates that, within the range of GDD and heat stress Oregon has experienced thus far, warmer conditions have generally corresponded with higher yields per acre across nearly every variety and region.

However, warmer seasons can raise yield while simultaneously degrading the flavor and quality characteristics that define Oregon’s premium wines, particularly Pinot Noir.

## The quality concern hiding under a healthy yield trend

In [ ]:
#| fig-cap: "Figure 7: Diurnal ripening temperature range, statewide average, 1981–2024, shown against the ideal 15–20°C range for Pinot Noir ripening"
sw_climate <- statewide %>%
  distinct(yr, diurnal_range_ripening) %>%
  arrange(yr)

ggplot(sw_climate, aes(yr, diurnal_range_ripening)) +
  annotate("rect", xmin = -Inf, xmax = Inf, ymin = 15, ymax = 20
           , fill = "#4A7CC7", alpha = 0.08) +
  annotate("text", x = min(sw_climate$yr, na.rm = TRUE), y = 20.3
           , label = "Ideal range for Pinot Noir ripening (15-20°C)"
           , hjust = 0, size = 3, color = "#4A7CC7") +
  geom_line(color = "grey60", alpha = 0.7, linewidth = 0.6) +
  geom_smooth(method = "lm", se = TRUE, color = "#8B1E3F", fill = "#8B1E3F", alpha = 0.15, linewidth = 1.1) +
  geom_vline(xintercept = 2020, linetype = "dashed", color = "grey40", linewidth = 0.5) +
  labs(title = "Ripening-season diurnal temperature range is shrinking"
       , subtitle = "Day/night temperature swing during ripening (Aug-Sep), statewide average, 1981–2024"
       , x = NULL, y = "Diurnal temperature range (°C)") +
  theme_minimal() +
  theme(plot.title = element_text(size = 14, face = "bold")
        , plot.subtitle = element_text(size = 9, color = "grey40"))

`geom_smooth()` using formula = 'y ~ x'

Figure 7 demonstrates a diurnal range that has narrowed from roughly 17°C in the 1980s to roughly 15°C today, indicating that nights are warming faster than days. This is consistent with a pattern documented globally in the climate literature and generally attributed to changes in cloud cover trapping heat overnight \[@ucs2022nights; @liu2024dtr\]. This day-night temperature swing is a key mechanism behind Oregon’s suitability for growing Pinot Noir: warm days build sugar, while cool nights slow the grape’s respiration long enough to retain the acidity, color, and aromatics that define the variety \[@kingestate2025; @ajev2012dtr\]. Controlled experiments on Pinot Noir specifically have found that compressing this range alters organic acid metabolism and can suppress anthocyanin accumulation, and a companion study found Pinot Noir was one of the varieties most sensitive to elevated diurnal temperature among the major red varieties tested, in contrast to Merlot, which showed little effect \[@frontiers2025bunchheating; @pmc2022highTemp\].

The narrowing diurnal range finding is a leading indicator worth flagging even though the problem has not yet surfaced in production data alone. This metric operates on wine quality through a physiological process that is more nuanced than heat-accumulation metrics alone. Thus, models built on yield alone will miss this leading indicator entirely. Future research could evaluate this risk through proxy indicators of wine quality such as price premiums or critic-score text mining.

## Predicting yield across the growing season

We next aim to create a predictive model that is built on two key observations from our analysis thus far: (1) yield has been resilient across the historical climate range and (2) the climate signal accumulates in stages over the course of the growing season—initial spring frost risk, then summer heat, then harvest-season conditions. We argue that model built only on a single end-of-season snapshot is less useful to a wine grower who experiences these climate conditions sequentially throughout the annual growth cycle.

# Conclusions

# References